# EDA - power consumption data
Lets look at this "consommation-quotidienne-brute-regionale" and "consommation-annuelle-brute-regionale" data from data.gouv.fr

In [1]:
import pandas as pd
import numpy as np
import duckdb

Global setup for DuckDB

In [2]:
con = duckdb.connect()

# load the table
def load_table(df, name):
    con.register(name, df)

# run the query
def run(query):
    return con.execute(query).df()

In [3]:
brute_path = "../Data/econsumption-daily-regionale.csv"
annual_path = "../Data/econsumption-annual-regionale.csv"

In [4]:
brute_raw = pd.read_csv(brute_path, sep=';')
annual_raw = pd.read_csv(annual_path, sep=';')

## Daily consumption

In [5]:
brute_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 2786679 entries, 0 to 2786678
Data columns (total 14 columns):
 #   Column                                        Dtype  
---  ------                                        -----  
 0   Date - Heure                                  str    
 1   Date                                          str    
 2   Heure                                         str    
 3   Code INSEE région                             int64  
 4   Région                                        str    
 5   Consommation brute gaz (MW PCS 0°C) - NaTran  float64
 6   Statut - NaTran                               str    
 7   Consommation brute gaz (MW PCS 0°C) - Teréga  float64
 8   Statut - Teréga                               str    
 9   Consommation brute gaz totale (MW PCS 0°C)    float64
 10  Consommation brute électricité (MW) - RTE     float64
 11  Statut - RTE                                  str    
 12  Consommation brute totale (MW)                float64
 13  flag_ign

There are GAS
1. NaTran
   - French natural gas transport network operator
   - handles gas transmission infrastructure

2. Teréga

   - another major gas transport operator in France
   - mostly southwest France network

This is Electricity


3.  RTE - electricity transmission operator in France

In [6]:
brute_raw.describe()

,Code INSEE région,Consommation brute gaz (MW PCS 0°C) - NaTran,Consommation brute gaz (MW PCS 0°C) - Teréga,Consommation brute gaz totale (MW PCS 0°C),Consommation brute électricité (MW) - RTE,Consommation brute totale (MW)
count,2.786679e+06,1.393268e+06,348336.000000,1.393267e+06,2.786676e+06,1.393267e+06
mean,4.991670e+01,3.903821e+03,1001.677843,4.154233e+03,4.409416e+03,8.564181e+03
std,2.564001e+01,3.624346e+03,1068.136841,3.528035e+03,2.121354e+03,5.288287e+03
min,1.100000e+01,0.000000e+00,1.000000,0.000000e+00,-3.239000e+03,1.175000e+03
25%,2.800000e+01,1.271000e+03,44.000000,1.673000e+03,2.732000e+03,4.890000e+03
50%,5.200000e+01,2.875000e+03,651.000000,3.184000e+03,4.052000e+03,7.280000e+03
75%,7.600000e+01,5.126000e+03,1655.000000,5.354000e+03,5.613000e+03,1.065200e+04
max,9.300000e+01,5.875800e+04,5746.000000,5.875800e+04,1.533800e+04,6.969900e+04


range or power consumption:

- min = -3239 and max = 15338 (MW) RTE
- total consommation - min 

In [7]:
brute_raw.isnull().sum()

Date - Heure                                          0
Date                                                  0
Heure                                                 0
Code INSEE région                                     0
Région                                                0
Consommation brute gaz (MW PCS 0°C) - NaTran    1393411
Statut - NaTran                                 1393407
Consommation brute gaz (MW PCS 0°C) - Teréga    2438343
Statut - Teréga                                 2438342
Consommation brute gaz totale (MW PCS 0°C)      1393412
Consommation brute électricité (MW) - RTE             3
Statut - RTE                                      35715
Consommation brute totale (MW)                  1393412
flag_ignore                                           0
dtype: int64

- Date - Heure
- Code INSEE région
- Région
- Consommation brute électricité (MW) - RTE

- Optional: Statut - RTE

In [8]:
brute_raw.columns

Index(['Date - Heure', 'Date', 'Heure', 'Code INSEE région', 'Région',
       'Consommation brute gaz (MW PCS 0°C) - NaTran', 'Statut - NaTran',
       'Consommation brute gaz (MW PCS 0°C) - Teréga', 'Statut - Teréga',
       'Consommation brute gaz totale (MW PCS 0°C)',
       'Consommation brute électricité (MW) - RTE', 'Statut - RTE',
       'Consommation brute totale (MW)', 'flag_ignore'],
      dtype='str')

- electricity data is clean and complete (RTE column works)
- gas data is split across NaTran and Teréga
- gas adds extra complexity (two providers, more nulls, more cleaning rules)

In [9]:
selected_brute_columns = [
    'Date - Heure',
    'Date', 
    'Heure', 
    'Code INSEE région', 
    'Région',
    'Consommation brute électricité (MW) - RTE', 
    'Statut - RTE',
    'Consommation brute totale (MW)'
]

In [11]:
cropped_data = brute_raw[selected_brute_columns]

In [12]:
print(cropped_data.shape)
cropped_data.head(5)

(2786679, 8)


,Date - Heure,Date,Heure,Code INSEE région,Région,Consommation brute électricité (MW) - RTE,Statut - RTE,Consommation brute totale (MW)
0,2013-06-05T04:00:00+00:00,2013-06-05,06:00,44,Grand Est,4556.0,Définitif,9556.0
1,2013-06-05T04:00:00+00:00,2013-06-05,06:00,93,Provence-Alpes-Côte d'Azur,3324.0,Définitif,6135.0
2,2013-06-05T04:30:00+00:00,2013-06-05,06:30,28,Normandie,2743.0,Définitif,NaN
3,2013-06-05T04:30:00+00:00,2013-06-05,06:30,32,Hauts-de-France,5194.0,Définitif,NaN
4,2013-06-05T04:30:00+00:00,2013-06-05,06:30,75,Nouvelle-Aquitaine,3687.0,Définitif,NaN


In [13]:
cropped_data['Statut - RTE'].unique()

<ArrowStringArray>
['Définitif', 'Consolidé', nan]
Length: 3, dtype: str

Meaning of Statut - RTE values:

- Définitif → final validated data, safe to use  **(highest trust)**
- Consolidé → corrected + cleaned but not final freeze **(normal trusted data)**
- NaN → missing status

In [30]:
cropped_data['Statut - RTE'].value_counts(dropna=False)

Statut - RTE
Définitif    1472244
Consolidé    1278720
NaN            35715
Name: count, dtype: int64

In [14]:
cropped_data['Région'].unique()

<ArrowStringArray>
[                 'Grand Est', 'Provence-Alpes-Côte d'Azur',
                  'Normandie',            'Hauts-de-France',
         'Nouvelle-Aquitaine',              'Île-de-France',
    'Bourgogne-Franche-Comté',           'Pays de la Loire',
        'Centre-Val de Loire',                   'Bretagne',
       'Auvergne-Rhône-Alpes',                  'Occitanie']
Length: 12, dtype: str

In [29]:
insee_reg_code = cropped_data['Code INSEE région'].unique()
print(len(insee_reg_code))
print(insee_reg_code)

12
[44 93 28 32 75 11 27 52 24 53 84 76]


In [15]:
load_table(cropped_data, "energy")

In [16]:
run("""
SELECT "Région", "Code INSEE région"
FROM energy
WHERE "Statut - RTE" = 'Définitif'
GROUP BY "Région", "Code INSEE région"
""")

,Région,Code INSEE région
0,Provence-Alpes-Côte d'Azur,93
1,Occitanie,76
2,Normandie,28
3,Centre-Val de Loire,24
4,Bretagne,53
5,Nouvelle-Aquitaine,75
6,Grand Est,44
7,Auvergne-Rhône-Alpes,84
8,Hauts-de-France,32
9,Pays de la Loire,52


In [17]:
cropped_data['Consommation brute totale (MW)'].isnull().sum()

np.int64(1393412)

In [18]:
run("""
SELECT 
    "Date - Heure",
    "Code INSEE région",
    COUNT(*) AS cnt
FROM energy
GROUP BY 1,2
HAVING COUNT(*) > 1
""")

,Date - Heure,Code INSEE région,cnt
0,2026-03-29T01:00:00+00:00,27,2
1,2023-03-26T01:00:00+00:00,11,2
2,2024-03-31T01:30:00+00:00,24,2
3,2015-03-29T01:30:00+00:00,11,2
4,2019-03-31T01:00:00+00:00,27,2
...,...,...,...
331,2018-03-25T01:00:00+00:00,28,2
332,2021-03-28T01:30:00+00:00,27,2
333,2014-03-30T01:00:00+00:00,44,2
334,2024-03-31T01:30:00+00:00,75,2


In [23]:
run("""
SELECT 
    "Date - Heure",
    "Code INSEE région",
    "Statut - RTE",
    COUNT(*) AS cnt
FROM energy
GROUP BY 1,2,3
HAVING COUNT(*) > 1
LIMIT 20
""")

,Date - Heure,Code INSEE région,Statut - RTE,cnt
0,2018-03-25T01:00:00+00:00,32,Définitif,2
1,2019-03-31T01:00:00+00:00,11,Définitif,2
2,2021-03-28T01:00:00+00:00,75,Consolidé,2
3,2013-03-31T01:00:00+00:00,75,Définitif,2
4,2026-03-29T01:00:00+00:00,76,NaN,2
5,2019-03-31T01:00:00+00:00,53,Définitif,2
6,2019-03-31T01:00:00+00:00,44,Définitif,2
7,2016-03-27T01:00:00+00:00,93,Définitif,2
8,2016-03-27T01:30:00+00:00,27,Définitif,2
9,2016-03-27T01:00:00+00:00,32,Définitif,2


In [19]:
# exact duplicate rows
cropped_data.duplicated().sum()

np.int64(0)

In [20]:
cropped_data[
    cropped_data.duplicated(
        subset=["Date - Heure", "Code INSEE région"],
        keep=False
    )
].sort_values(
    ["Date - Heure", "Code INSEE région"]
)

,Date - Heure,Date,Heure,Code INSEE région,Région,Consommation brute électricité (MW) - RTE,Statut - RTE,Consommation brute totale (MW)
1149029,2013-03-31T01:00:00+00:00,2013-03-31,02:00,11,Île-de-France,9468.0,Définitif,12410.0
2403779,2013-03-31T01:00:00+00:00,2013-03-31,03:00,11,Île-de-France,9468.0,Définitif,12410.0
1149033,2013-03-31T01:00:00+00:00,2013-03-31,03:00,24,Centre-Val de Loire,2737.0,Définitif,5803.0
2403773,2013-03-31T01:00:00+00:00,2013-03-31,02:00,24,Centre-Val de Loire,2737.0,Définitif,5831.0
152547,2013-03-31T01:00:00+00:00,2013-03-31,02:00,27,Bourgogne-Franche-Comté,2588.0,Définitif,6070.0
...,...,...,...,...,...,...,...,...
2226728,2026-03-29T01:30:00+00:00,2026-03-29,03:30,76,Occitanie,4503.0,NaN,NaN
1671667,2026-03-29T01:30:00+00:00,2026-03-29,02:30,84,Auvergne-Rhône-Alpes,7888.0,NaN,NaN
2226729,2026-03-29T01:30:00+00:00,2026-03-29,03:30,84,Auvergne-Rhône-Alpes,7856.0,NaN,NaN
554090,2026-03-29T01:30:00+00:00,2026-03-29,03:30,93,Provence-Alpes-Côte d'Azur,4468.0,NaN,NaN


In [21]:
cropped_data.duplicated(subset=["Date - Heure", "Code INSEE région"]).sum()

np.int64(336)

#### per day useage

In [22]:
run("""
SELECT
    Date,
    Région,
    SUM("Consommation brute électricité (MW) - RTE") * 0.5 AS daily_mwh
FROM energy
GROUP BY Date, Région
ORDER BY Date
""")

,Date,Région,daily_mwh
0,2013-01-01,Auvergne-Rhône-Alpes,177679.5
1,2013-01-01,Nouvelle-Aquitaine,116570.0
2,2013-01-01,Grand Est,109659.0
3,2013-01-01,Bretagne,60989.5
4,2013-01-01,Hauts-de-France,126914.5
...,...,...,...
58051,2026-03-31,Occitanie,115919.0
58052,2026-03-31,Centre-Val de Loire,57205.0
58053,2026-03-31,Normandie,77307.5
58054,2026-03-31,Grand Est,134083.0


In [25]:
cofn=cropped_data.groupby(['Date', 'Heure', 'Code INSEE région']).size()
cofn.unique()

array([1])

In [27]:
print(cropped_data.Date.min())
print(cropped_data.Date.max())

2013-01-01
2026-03-31


## Anual Consumption


In [31]:
annual_raw.columns

Index(['Année', 'Région', 'Code INSEE région', 'Statut - Teréga',
       'Consommation brute électricité (GWh) - RTE', 'Statut - RTE',
       'Géo-shape région', 'Géo-point région',
       'Consommation brute gaz (GWh PCS 0°C) - NaTran', 'Statut - NaTran',
       'Consommation brute gaz (GWh PCS 0°C) - Teréga',
       'Consommation brute gaz totale (GWh PCS 0°C)',
       'Consommation brute totale (GWh)'],
      dtype='str')

In [32]:
annual_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 169 entries, 0 to 168
Data columns (total 13 columns):
 #   Column                                         Non-Null Count  Dtype  
---  ------                                         --------------  -----  
 0   Année                                          169 non-null    int64  
 1   Région                                         169 non-null    str    
 2   Code INSEE région                              169 non-null    int64  
 3   Statut - Teréga                                39 non-null     str    
 4   Consommation brute électricité (GWh) - RTE     169 non-null    int64  
 5   Statut - RTE                                   169 non-null    str    
 6   Géo-shape région                               169 non-null    str    
 7   Géo-point région                               169 non-null    str    
 8   Consommation brute gaz (GWh PCS 0°C) - NaTran  156 non-null    float64
 9   Statut - NaTran                                161 non-null    st

In [7]:
annual_raw.isnull().sum()

Année                                              0
Région                                             0
Code INSEE région                                  0
Statut - Teréga                                  130
Consommation brute électricité (GWh) - RTE         0
Statut - RTE                                       0
Géo-shape région                                   0
Géo-point région                                   0
Consommation brute gaz (GWh PCS 0°C) - NaTran     13
Statut - NaTran                                    8
Consommation brute gaz (GWh PCS 0°C) - Teréga    130
Consommation brute gaz totale (GWh PCS 0°C)       13
Consommation brute totale (GWh)                   13
dtype: int64

In [33]:
annual_raw.describe()

,Année,Code INSEE région,Consommation brute électricité (GWh) - RTE,Consommation brute gaz (GWh PCS 0°C) - NaTran,Consommation brute gaz (GWh PCS 0°C) - Teréga,Consommation brute gaz totale (GWh PCS 0°C),Consommation brute totale (GWh)
count,169.000000,169.000000,169.000000,156.000000,39.000000,156.000000,156.000000
mean,2019.000000,53.307692,35764.532544,34268.480769,8714.461538,36447.096154,75003.801282
std,3.752777,27.372649,18287.479878,22178.109304,6193.901236,20224.683403,34084.418548
min,2013.000000,11.000000,2100.000000,4160.000000,186.000000,11881.000000,29171.000000
25%,2016.000000,28.000000,21851.000000,15694.250000,279.500000,18765.500000,43102.500000
50%,2019.000000,52.000000,37429.000000,24537.000000,11583.000000,28776.500000,68072.500000
75%,2022.000000,76.000000,46092.000000,50544.750000,13738.500000,50816.250000,108876.500000
max,2025.000000,94.000000,75705.000000,80983.000000,15924.000000,80983.000000,154975.000000


In [34]:
selected_annual_columns = ['Année',
                            'Région',
                            'Code INSEE région',
                            'Consommation brute électricité (GWh) - RTE', 
                            'Statut - RTE',
                            'Consommation brute totale (GWh)']

In [35]:
cropped_annual = annual_raw[selected_annual_columns]

In [37]:
load_table(cropped_annual,'cropped_annual')

In [38]:
cropped_annual.isnull().sum()

Année                                          0
Région                                         0
Code INSEE région                              0
Consommation brute électricité (GWh) - RTE     0
Statut - RTE                                   0
Consommation brute totale (GWh)               13
dtype: int64

In [39]:
cropped_annual.Année.max()

np.int64(2025)

In [40]:
cropped_annual['Région'].unique()

<ArrowStringArray>
[             'Île-de-France',        'Centre-Val de Loire',
    'Bourgogne-Franche-Comté',                  'Normandie',
            'Hauts-de-France',                  'Grand Est',
           'Pays de la Loire',                   'Bretagne',
         'Nouvelle-Aquitaine',                  'Occitanie',
       'Auvergne-Rhône-Alpes', 'Provence-Alpes-Côte d'Azur',
                      'Corse']
Length: 13, dtype: str

In [45]:
insee_reg_code_a = cropped_annual['Code INSEE région'].unique()
print(insee_reg_code_a)
print(len(insee_reg_code_a))

[11 24 27 28 32 44 52 53 75 76 84 93 94]
13


In [47]:
cropped_annual['Année'].min(), cropped_annual['Année'].max()


(np.int64(2013), np.int64(2025))

In [48]:
cropped_annual['Statut - RTE'].value_counts()

Statut - RTE
Définitif    156
Consolidé     13
Name: count, dtype: int64